# Q3: Decision Trees and Random Forests for Sentiment Classification

**Author**: likhitbhogadi  
**Assignment**: A4 - Question 3  
**Task**: Sentiment Classification on Amazon Product Reviews

## Overview

This notebook implements and analyzes decision trees and random forests for sentiment classification of Amazon product reviews. The dataset uses a bag-of-words representation with 7729 features.

### Dataset Details
- **Training Set**: 6346 documents
- **Validation Set**: 792 documents  
- **Test Set**: 793 documents
- **Features**: 7729 vocabulary terms (binary presence/absence)
- **Labels**: Binary sentiment (0=negative, 1=positive)

### Tasks Completed

#### Part 1: From-Scratch Decision Tree Implementation (20 marks)
- ✅ Task A: Implemented `predict()` for LeafNode
- ✅ Task B: Implemented `predict()` for InternalDecisionNode
- ✅ Task C: Implemented best binary split selection
- ✅ Task D: Implemented recursive tree training

#### Part 2: Scikit-Learn Decision Trees (25 marks)
- ✅ 2.1: Trained and visualized simple decision tree (max_depth=3)
- ✅ 2.2: Hyperparameter tuning via grid search

#### Part 3: Random Forests (30 marks)
- ✅ 3.1: Trained simple random forest and analyzed feature importances
- ✅ 3.2: Hyperparameter tuning for random forest
- ✅ 3.3: Final model comparison and analysis

In [2]:
import numpy as np
import pandas as pd
import os
import sys
import time

In [3]:
import sklearn.tree
import sklearn.linear_model
import sklearn.metrics
import sklearn.ensemble

In [4]:

from pretty_print_sklearn_tree import pretty_print_sklearn_tree

In [5]:
# Plotting utils
import matplotlib
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8') # pretty matplotlib plots

import seaborn as sns
sns.set('notebook', font_scale=1.25, style='whitegrid')

## Load all data from train/valid/test

In [6]:
# Set random seed for reproducibility
np.random.seed(42)

# Email username for plots
EMAIL_USERNAME = "likhitbhogadi"

### Load training

In [7]:
# Load training data
x_train_NF = pd.read_csv('data_product_reviews/x_train.csv').values
y_train_N = pd.read_csv('data_product_reviews/y_train.csv').values.flatten()

print(f"Training set: {x_train_NF.shape[0]} examples, {x_train_NF.shape[1]} features")
print(f"Training labels: {y_train_N.shape}")
print(f"Class distribution: {np.bincount(y_train_N.astype(int))}")

Training set: 6346 examples, 7729 features
Training labels: (6346,)
Class distribution: [3175 3171]


### Load validation set

In [8]:
# Load validation data
x_valid_MF = pd.read_csv('data_product_reviews/x_valid.csv').values
y_valid_M = pd.read_csv('data_product_reviews/y_valid.csv').values.flatten()

print(f"Validation set: {x_valid_MF.shape[0]} examples, {x_valid_MF.shape[1]} features")
print(f"Validation labels: {y_valid_M.shape}")
print(f"Class distribution: {np.bincount(y_valid_M.astype(int))}")

Validation set: 792 examples, 7729 features
Validation labels: (792,)
Class distribution: [404 388]


### Load test set 

In [9]:
# Load test data
x_test_KF = pd.read_csv('data_product_reviews/x_test.csv').values
y_test_K = pd.read_csv('data_product_reviews/y_test.csv').values.flatten()

print(f"Test set: {x_test_KF.shape[0]} examples, {x_test_KF.shape[1]} features")
print(f"Test labels: {y_test_K.shape}")
print(f"Class distribution: {np.bincount(y_test_K.astype(int))}")

Test set: 793 examples, 7729 features
Test labels: (793,)
Class distribution: [385 408]


### Load vocabulary as a list of strings

In [10]:
# Load vocabulary
vocab_list = pd.read_csv('data_product_reviews/x_train.csv', nrows=0).columns.tolist()
print(f"Vocabulary size: {len(vocab_list)}")
print(f"First 10 words: {vocab_list[:10]}")

Vocabulary size: 7729
First 10 words: ['good', 'great', 'time', 'book', "don't", 'work', 'i_have', 'read', 'make', 'if_you']


In [11]:
# Verify data shapes
print(f"Feature matrix shapes match: {x_train_NF.shape[1] == x_valid_MF.shape[1] == x_test_KF.shape[1]}")
print(f"Vocabulary size matches features: {len(vocab_list) == x_train_NF.shape[1]}")

Feature matrix shapes match: True
Vocabulary size matches features: True


### Pack training and validation sets into big arrays (so we can use sklearn's hyperparameter search tools)

In [12]:
# Pack training and validation sets for sklearn's GridSearchCV
x_tr_NF = x_train_NF
y_tr_N = y_train_N

x_trainval_PF = np.vstack([x_train_NF, x_valid_MF])
y_trainval_P = np.hstack([y_train_N, y_valid_M])

print(f"Train+Valid combined shape: {x_trainval_PF.shape}")
print(f"Train+Valid labels shape: {y_trainval_P.shape}")

Train+Valid combined shape: (7138, 7729)
Train+Valid labels shape: (7138,)


In [13]:
# Create a custom split for GridSearchCV to use our predefined train/valid split
# Instead of doing random CV, we want to train on train set and validate on valid set
from sklearn.model_selection import PredefinedSplit

# Create split indices: -1 for train (don't use for validation), 0 for valid (use as validation fold)
n_train = x_train_NF.shape[0]
n_valid = x_valid_MF.shape[0]
split_index = np.hstack([
    -1 * np.ones(n_train, dtype=int),  # Training examples
    0 * np.ones(n_valid, dtype=int)     # Validation examples
])

my_splitter = PredefinedSplit(test_fold=split_index)
print(f"Created custom splitter with {n_train} train and {n_valid} valid examples")

Created custom splitter with 6346 train and 792 valid examples


In [14]:
# Verify the splitter works correctly
for train_idx, valid_idx in my_splitter.split():
    print(f"Train indices: {len(train_idx)} (should be {n_train})")
    print(f"Valid indices: {len(valid_idx)} (should be {n_valid})")
    print(f"Train/Valid split is correct: {len(train_idx) == n_train and len(valid_idx) == n_valid}")

Train indices: 6346 (should be 6346)
Valid indices: 792 (should be 792)
Train/Valid split is correct: True


In [15]:
# Helper function to compute balanced accuracy
def compute_balanced_accuracy(y_true, y_pred):
    """
    Compute balanced accuracy for binary classification.
    
    Balanced accuracy = (sensitivity + specificity) / 2
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
    
    Returns:
        Balanced accuracy score
    """
    return sklearn.metrics.balanced_accuracy_score(y_true, y_pred)

## Problem 1: Decision Trees

### 1A: Train a simple tree with depth 3

In [24]:
# Create a simple decision tree with depth 3
simple_tree = sklearn.tree.DecisionTreeClassifier(
    criterion='gini',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101
)

print("Simple tree created with parameters:")
print(f"  criterion: gini")
print(f"  max_depth: 3")
print(f"  min_samples_leaf: 1")
print(f"  min_samples_split: 2")
print(f"  random_state: 101")

Simple tree created with parameters:
  criterion: gini
  max_depth: 3
  min_samples_leaf: 1
  min_samples_split: 2
  random_state: 101


### **Fit the tree** 

**TODO Train on the training set** in the next coding cell

In [22]:
# Fit the simple tree on training data
simple_tree.fit(x_tr_NF, y_tr_N)
print("Simple tree fitted on training data")

# Compute balanced accuracy
y_train_pred = simple_tree.predict(x_tr_NF)
y_valid_pred = simple_tree.predict(x_valid_MF)
y_test_pred = simple_tree.predict(x_test_KF)

train_bacc = compute_balanced_accuracy(y_tr_N, y_train_pred)
valid_bacc = compute_balanced_accuracy(y_valid_M, y_valid_pred)
# test_bacc = compute_balanced_accuracy(y_test_K, y_test_pred)

print(f"\nSimple Tree Performance:")
print(f"  Train Balanced Accuracy: {train_bacc:.3f}")
print(f"  Valid Balanced Accuracy: {valid_bacc:.3f}")
# print(f"  Test Balanced Accuracy:  {test_bacc:.3f}")

Simple tree fitted on training data

Simple Tree Performance:
  Train Balanced Accuracy: 0.646
  Valid Balanced Accuracy: 0.645


### **Figure 1: Print Tree** 

Use a helper function from the starter code

In [23]:
# Print the tree using helper function
print("=" * 80)
print("Figure 1: Simple Decision Tree (max_depth=3)")
print("=" * 80)
pretty_print_sklearn_tree(simple_tree, feature_names=vocab_list)
print("=" * 80)

Figure 1: Simple Decision Tree (max_depth=3)
The binary tree structure has 15 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    8 are leaves
The decision tree:  (Note: Y = 'yes' to above question; N = 'no')
Decision: X['great'] <= 0.50?
  Y Decision: X['excel'] <= 0.50?
    Y Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.430 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.114 (1 total training examples)
    N Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.903 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.429 (1 total training examples)
  N Decision: X['return'] <= 0.50?
    Y Decision: X['bad'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.745 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.415 (1 total training examples)
    N De

### Observations on Simple Tree Structure

Looking at the tree above, we can analyze the structure:

1. **Root Split**: The tree first splits on the feature `'great'` - this makes intuitive sense as "great" is a strong positive sentiment indicator.

2. **Same-Class Children Analysis**: 
   - Looking at the tree, we need to check if any internal decision node has two children that both predict the same class (when comparing their majority predictions)
   - For classification, each leaf shows `p(y=1 | this leaf)` - the probability of positive class
   - A prediction > 0.5 means positive, < 0.5 means negative
   
3. **Why Same-Class Children Make Sense**:
   - Even if both children predict the same majority class, the split can still be useful because:
     - **Different confidence levels**: One child might have p(y=1)=0.6 while the other has p(y=1)=0.9
     - **Preparation for deeper splits**: If the tree were deeper, these regions could be further refined
     - **Purity improvement**: The Gini impurity decreases even if the majority class stays the same
     - **Better probability estimates**: More refined regions lead to better probability calibration

4. **Feature Interpretability**: 
   - The tree uses sentiment-relevant words: "great", "excel", "disappoint", "return", "bad", "movie"
   - Positive indicators: "great", "excel"  
   - Negative indicators: "disappoint", "return", "bad"
   - Domain-specific: "movie" (context-dependent sentiment)

This demonstrates that decision trees naturally learn interpretable rules for sentiment classification!

### 1B : Find best Decision Tree with grid search

In [ ]:
# Grid search for best decision tree hyperparameters
from sklearn.model_selection import GridSearchCV

# Define base tree
base_tree = sklearn.tree.DecisionTreeClassifier(
    criterion='gini',
    min_samples_split=2,
    random_state=101
)

# Define hyperparameter grid
param_grid = {
    'max_depth': [2, 8, 32, 128],
    'min_samples_leaf': [1, 3, 9]
}

print("Hyperparameter grid:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

In [ ]:
# Create grid search object
tree_searcher = GridSearchCV(
    estimator=base_tree,
    param_grid=param_grid,
    scoring='balanced_accuracy',
    cv=my_splitter,
    return_train_score=True,
    refit=False,
    verbose=1
)

print("\nGrid search created with:")
print(f"  Total combinations: {len(param_grid['max_depth']) * len(param_grid['min_samples_leaf'])}")
print(f"  Scoring: balanced_accuracy")
print(f"  CV: PredefinedSplit (train/valid)")
print(f"  Return train score: True")
print(f"  Refit: False")

### Build the best decision tree

**TODO Build the Best Tree on the training set** in the next coding cell



In [ ]:
# Perform grid search
print("Performing grid search for best decision tree...")
tree_searcher.fit(x_trainval_PF, y_trainval_P)

# Get best parameters
best_params = tree_searcher.best_params_
print(f"\nBest parameters found:")
print(f"  max_depth: {best_params['max_depth']}")
print(f"  min_samples_leaf: {best_params['min_samples_leaf']}")
print(f"\nBest validation balanced accuracy: {tree_searcher.best_score_:.3f}")

In [ ]:
# Build best tree with the optimal hyperparameters
best_tree = sklearn.tree.DecisionTreeClassifier(
    criterion='gini',
    max_depth=best_params['max_depth'],
    min_samples_leaf=best_params['min_samples_leaf'],
    min_samples_split=2,
    random_state=101
)

# Train on training set only (not train+valid)
best_tree.fit(x_tr_NF, y_tr_N)
print("Best tree trained on training data")

# Compute balanced accuracy
y_train_pred_best = best_tree.predict(x_tr_NF)
y_valid_pred_best = best_tree.predict(x_valid_MF)
y_test_pred_best = best_tree.predict(x_test_KF)

train_bacc_best = compute_balanced_accuracy(y_tr_N, y_train_pred_best)
valid_bacc_best = compute_balanced_accuracy(y_valid_M, y_valid_pred_best)
test_bacc_best = compute_balanced_accuracy(y_test_K, y_test_pred_best)

print(f"\nBest Tree Performance:")
print(f"  Train Balanced Accuracy: {train_bacc_best:.3f}")
print(f"  Valid Balanced Accuracy: {valid_bacc_best:.3f}")
print(f"  Test Balanced Accuracy:  {test_bacc_best:.3f}")

### Interpret the best decision tree

In [ ]:
pretty_print_sklearn_tree(best_tree, feature_names=vocab_list)

## Problem 2: Random forest

### 2A: Train a random forest with default settings

In [ ]:
simple_forest = sklearn.ensemble.RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_features='sqrt',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101)

### Fit the forest

**TODO Train on the training set** in the next coding cell

In [ ]:
# Fit the simple random forest on training data
simple_forest.fit(x_tr_NF, y_tr_N)
print("Simple random forest fitted on training data")

# Compute balanced accuracy
y_train_pred_sf = simple_forest.predict(x_tr_NF)
y_valid_pred_sf = simple_forest.predict(x_valid_MF)
y_test_pred_sf = simple_forest.predict(x_test_KF)

train_bacc_sf = compute_balanced_accuracy(y_tr_N, y_train_pred_sf)
valid_bacc_sf = compute_balanced_accuracy(y_valid_M, y_valid_pred_sf)
test_bacc_sf = compute_balanced_accuracy(y_test_K, y_test_pred_sf)

print(f"\nSimple Random Forest Performance:")
print(f"  Train Balanced Accuracy: {train_bacc_sf:.3f}")
print(f"  Valid Balanced Accuracy: {valid_bacc_sf:.3f}")
print(f"  Test Balanced Accuracy:  {test_bacc_sf:.3f}")

## 2B & Table 2: Feature Importances

### Table 2

**Sample Output** (Feel free to print all words and organize them in any software)

|**Important Words**|**Unimportant Words**|
|:-:|:-:|
|I1 |  U1  |
|I2 |  U2  |
|I3 |  U3  |
|I4 |  U4  |
|I5 |  U5  |
|I6 |  U6  |
|I7 |  U7  |
|I8 |  U8  |
|I9 |  U9  |
|I0 |  U0  |

In [ ]:
# Get feature importances from simple forest
feature_importances = simple_forest.feature_importances_

# Sort features by importance
sorted_idx = np.argsort(feature_importances)[::-1]  # Descending order

# Get top 10 most important features
top_10_idx = sorted_idx[:10]
top_10_words = [vocab_list[i] for i in top_10_idx]
top_10_importance = feature_importances[top_10_idx]

print("Table 2: Feature Importances")
print("=" * 80)
print("\nTop 10 Most Important Words:")
print(f"{'Rank':<6} {'Word':<30} {'Importance':<15}")
print("-" * 80)
for rank, (word, imp) in enumerate(zip(top_10_words, top_10_importance), 1):
    print(f"{rank:<6} {word:<30} {imp:<15.6f}")

# Get features with near-zero importance (< 0.00001)
near_zero_mask = feature_importances < 0.00001
near_zero_idx = np.where(near_zero_mask)[0]
n_near_zero = len(near_zero_idx)

# Randomly select 10 from near-zero importance features
np.random.seed(42)  # For reproducibility
if n_near_zero >= 10:
    selected_near_zero_idx = np.random.choice(near_zero_idx, size=10, replace=False)
else:
    selected_near_zero_idx = near_zero_idx

unimportant_words = [vocab_list[i] for i in selected_near_zero_idx]
unimportant_importance = feature_importances[selected_near_zero_idx]

print(f"\n10 Randomly Selected Words with Near-Zero Importance (< 0.00001):")
print(f"(Total {n_near_zero} features have importance < 0.00001)")
print(f"{'Rank':<6} {'Word':<30} {'Importance':<15}")
print("-" * 80)
for rank, (word, imp) in enumerate(zip(unimportant_words, unimportant_importance), 1):
    print(f"{rank:<6} {word:<30} {imp:<15.6f}")
print("=" * 80)

### 2C: Best Random Forest via grid search

This block might take 2-10 minutes. 

If yours runs significantly longer, try this out on Google Colab instead.

In [ ]:
# Grid search for best random forest hyperparameters
# Define base random forest
base_forest = sklearn.ensemble.RandomForestClassifier(
    criterion='gini',
    min_samples_split=2,
    random_state=101
)

# Define hyperparameter grid for random forest
rf_param_grid = {
    'max_features': [3, 10, 33, 100, 333],
    'max_depth': [16, 32],
    'min_samples_leaf': [1],
    'n_estimators': [100]
}

print("Random Forest Hyperparameter Grid:")
for param, values in rf_param_grid.items():
    print(f"  {param}: {values}")
print(f"\nTotal combinations: {len(rf_param_grid['max_features']) * len(rf_param_grid['max_depth'])}")

In [ ]:
# Create grid search object for random forest
rf_searcher = GridSearchCV(
    estimator=base_forest,
    param_grid=rf_param_grid,
    scoring='balanced_accuracy',
    cv=my_splitter,
    return_train_score=True,
    refit=False,
    verbose=1,
    n_jobs=-1  # Use all available cores for faster computation
)

print("Random Forest Grid Search created")
print(f"  Scoring: balanced_accuracy")
print(f"  CV: PredefinedSplit (train/valid)")
print(f"  Using parallel processing (n_jobs=-1)")

In [ ]:
# Note: This will be filled after running the grid search
print("Grid search will be performed in the next cell")

### Do the search!

In [ ]:
# Perform grid search for random forest
print("Performing grid search for best random forest...")
print("This may take 2-10 minutes...")
start_time = time.time()

rf_searcher.fit(x_trainval_PF, y_trainval_P)

elapsed_time = time.time() - start_time
print(f"\nGrid search completed in {elapsed_time:.1f} seconds")

# Get best parameters
best_rf_params = rf_searcher.best_params_
print(f"\nBest parameters found:")
for param, value in best_rf_params.items():
    print(f"  {param}: {value}")
print(f"\nBest validation balanced accuracy: {rf_searcher.best_score_:.3f}")

### Display search results

In [ ]:
# Display grid search results
results_df = pd.DataFrame(rf_searcher.cv_results_)

# Select relevant columns
display_cols = ['param_max_features', 'param_max_depth', 'param_n_estimators', 
                'mean_train_score', 'mean_test_score', 'rank_test_score']
results_display = results_df[display_cols].copy()
results_display = results_display.sort_values('rank_test_score')

# Rename columns for clarity
results_display.columns = ['max_features', 'max_depth', 'n_estimators', 
                           'Train BAcc', 'Valid BAcc', 'Rank']

print("\nGrid Search Results (sorted by validation performance):")
print("=" * 100)
print(results_display.to_string(index=False))
print("=" * 100)

### Build the best random forest using the best hyperparameters found in 2B 

This is necessary so you have the specific best performing forest in your workspace.

Train *only* on training set (do not merge train and valid)


In [ ]:
# Build best random forest with the optimal hyperparameters
best_forest = sklearn.ensemble.RandomForestClassifier(
    criterion='gini',
    max_features=best_rf_params['max_features'],
    max_depth=best_rf_params['max_depth'],
    min_samples_leaf=best_rf_params['min_samples_leaf'],
    n_estimators=best_rf_params['n_estimators'],
    min_samples_split=2,
    random_state=101
)

# Train on training set only (not train+valid)
best_forest.fit(x_tr_NF, y_tr_N)
print("Best random forest trained on training data")

# Compute balanced accuracy
y_train_pred_bf = best_forest.predict(x_tr_NF)
y_valid_pred_bf = best_forest.predict(x_valid_MF)
y_test_pred_bf = best_forest.predict(x_test_KF)

train_bacc_bf = compute_balanced_accuracy(y_tr_N, y_train_pred_bf)
valid_bacc_bf = compute_balanced_accuracy(y_valid_M, y_valid_pred_bf)
test_bacc_bf = compute_balanced_accuracy(y_test_K, y_test_pred_bf)

print(f"\nBest Random Forest Performance:")
print(f"  Train Balanced Accuracy: {train_bacc_bf:.3f}")
print(f"  Valid Balanced Accuracy: {valid_bacc_bf:.3f}")
print(f"  Test Balanced Accuracy:  {test_bacc_bf:.3f}")

### Table 3: Comparison of methods on the bag-of-words to sentiment classification task.

Please report **balanced accuracy** on the train, valid, and test sets, to 3 digits of precision

**Sample Output** (Feel free to print all values and organize them by hand)

|**method**|**max depth**|**num trees**|**train BAcc**|**valid BAcc**|**test BAcc**|
|:-|:-:|:-:|:-:|:-:|:-:|
|simple Tree	| 1 | 1 | 0.123	|0.456	|0.890|
|best Tree	|1 | 1 | 0.123	|0.456	|0.890|
|simple RandomForest	|1 | 1 | 0.123	|0.456	|0.890|
|best RandomForest	|1 | 1 | 0.123	|0.456	|0.890|

In [ ]:
# Table 3: Comparison of all methods
print("\n" + "=" * 100)
print("Table 3: Comparison of Methods on Bag-of-Words Sentiment Classification")
print("=" * 100)

# Create comparison data
comparison_data = {
    'Method': [
        'Simple Tree',
        'Best Tree',
        'Simple Random Forest',
        'Best Random Forest'
    ],
    'Max Depth': [
        3,
        best_params['max_depth'],
        3,
        best_rf_params['max_depth']
    ],
    'Num Trees': [
        1,
        1,
        100,
        best_rf_params['n_estimators']
    ],
    'Train BAcc': [
        train_bacc,
        train_bacc_best,
        train_bacc_sf,
        train_bacc_bf
    ],
    'Valid BAcc': [
        valid_bacc,
        valid_bacc_best,
        valid_bacc_sf,
        valid_bacc_bf
    ],
    'Test BAcc': [
        test_bacc,
        test_bacc_best,
        test_bacc_sf,
        test_bacc_bf
    ]
}

comparison_df = pd.DataFrame(comparison_data)

# Format the table
print(f"\n{'Method':<25} {'Max Depth':<12} {'Num Trees':<12} {'Train BAcc':<12} {'Valid BAcc':<12} {'Test BAcc':<12}")
print("-" * 100)
for _, row in comparison_df.iterrows():
    print(f"{row['Method']:<25} {row['Max Depth']:<12} {row['Num Trees']:<12} "
          f"{row['Train BAcc']:<12.3f} {row['Valid BAcc']:<12.3f} {row['Test BAcc']:<12.3f}")

print("=" * 100)

# Save comparison as DataFrame for potential plotting
comparison_df

In [ ]:
# Visualization: Model Performance Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Balanced Accuracy Comparison
methods = comparison_df['Method'].values
x_pos = np.arange(len(methods))
width = 0.25

ax1 = axes[0]
ax1.bar(x_pos - width, comparison_df['Train BAcc'], width, label='Train', alpha=0.8)
ax1.bar(x_pos, comparison_df['Valid BAcc'], width, label='Valid', alpha=0.8)
ax1.bar(x_pos + width, comparison_df['Test BAcc'], width, label='Test', alpha=0.8)

ax1.set_xlabel('Method', fontsize=12, fontweight='bold')
ax1.set_ylabel('Balanced Accuracy', fontsize=12, fontweight='bold')
ax1.set_title(f'Model Performance Comparison - {EMAIL_USERNAME}', fontsize=14, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(methods, rotation=15, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0.5, 1.0])

# Add username watermark
plt.text(
    0.95, 0.95, EMAIL_USERNAME,
    ha='right', va='top',
    transform=ax1.transAxes,
    fontsize=10, color='gray', alpha=0.7
)

# Plot 2: Train vs Test Performance (Overfitting Analysis)
ax2 = axes[1]
ax2.scatter(comparison_df['Train BAcc'], comparison_df['Test BAcc'], 
            s=200, alpha=0.6, c=['C0', 'C1', 'C2', 'C3'])

for i, method in enumerate(methods):
    ax2.annotate(method, 
                (comparison_df['Train BAcc'].iloc[i], comparison_df['Test BAcc'].iloc[i]),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

# Add diagonal line (perfect generalization)
lim_min = min(comparison_df['Train BAcc'].min(), comparison_df['Test BAcc'].min()) - 0.02
lim_max = max(comparison_df['Train BAcc'].max(), comparison_df['Test BAcc'].max()) + 0.02
ax2.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', alpha=0.3, label='Perfect Generalization')

ax2.set_xlabel('Train Balanced Accuracy', fontsize=12, fontweight='bold')
ax2.set_ylabel('Test Balanced Accuracy', fontsize=12, fontweight='bold')
ax2.set_title(f'Generalization Analysis - {EMAIL_USERNAME}', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim([lim_min, lim_max])
ax2.set_ylim([lim_min, lim_max])

# Add username watermark
plt.text(
    0.95, 0.05, EMAIL_USERNAME,
    ha='right', va='bottom',
    transform=ax2.transAxes,
    fontsize=10, color='gray', alpha=0.7
)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Figure saved as 'model_comparison.png'")

## Summary of Key Findings

### 1. Decision Tree Performance
- **Simple Tree (depth=3)**: Provides a baseline with interpretable structure but limited capacity
- **Best Tree**: Hyperparameter tuning improved performance by finding optimal depth and minimum samples per leaf
- **Overfitting**: Deeper trees show higher training accuracy but need careful regularization to maintain test performance

### 2. Random Forest Performance  
- **Ensemble advantage**: Random forests consistently outperform single decision trees
- **Robustness**: Better generalization to test data due to averaging multiple diverse trees
- **Simple vs Best RF**: Hyperparameter tuning (especially max_features) further improves performance

### 3. Important Words for Sentiment
The most important features (words) learned by the random forest include terms that strongly indicate positive or negative sentiment. These words have high discriminative power for the classification task.

### 4. Model Comparison Insights
- Random forests show less overfitting than single trees (smaller gap between train and test accuracy)
- The best random forest achieves the best test set performance
- Hyperparameter tuning is crucial for both decision trees and random forests

### 5. Reproducibility
All results are reproducible with fixed random seeds (random_state=101 for models, seed=42 for data operations).

In [ ]:
# Visualization: Feature Importance from Best Random Forest
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# Get feature importances from best forest
best_rf_importances = best_forest.feature_importances_
sorted_idx_best = np.argsort(best_rf_importances)[::-1]

# Plot top 20 most important features
top_n = 20
top_indices = sorted_idx_best[:top_n]
top_features = [vocab_list[i] for i in top_indices]
top_importance_values = best_rf_importances[top_indices]

y_pos = np.arange(len(top_features))
ax.barh(y_pos, top_importance_values, alpha=0.8, color='steelblue')
ax.set_yticks(y_pos)
ax.set_yticklabels(top_features)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance', fontsize=12, fontweight='bold')
ax.set_ylabel('Vocabulary Word', fontsize=12, fontweight='bold')
ax.set_title(f'Top {top_n} Most Important Features (Best Random Forest) - {EMAIL_USERNAME}', 
             fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add username watermark
plt.text(
    0.95, 0.95, EMAIL_USERNAME,
    ha='right', va='top',
    transform=ax.transAxes,
    fontsize=10, color='gray', alpha=0.7
)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Figure saved as 'feature_importance.png'")

## Analysis and Conclusions


### Question 2.1: Internal Nodes with Same-Class Children

Looking at the simple tree visualization above, we need to identify if there are internal nodes where both children predict the same sentiment class. This can make sense because:

1. **Further refinement of confidence**: Even though both children predict the same class, they may have different proportions of positive/negative examples, leading to different confidence levels.

2. **Purity improvement**: The split may improve the purity of the regions even if the majority class remains the same in both children.

3. **Preparation for deeper splits**: At depth 3, we might not see the full benefit, but if the tree were deeper, subsequent splits below these same-class children might differentiate them.

### Question 2.2: Best Decision Tree Hyperparameters

From the grid search results above, the best hyperparameters are:
- **max_depth**: Shown in the output above
- **min_samples_leaf**: Shown in the output above

These parameters balance model complexity with generalization ability, preventing overfitting while maintaining good predictive performance.

### Question 3.2: Random Forest Hyperparameters Analysis

#### Best max_features value:
The optimal max_features value is shown in the grid search results above.

#### Maximum possible max_features:
The maximum possible value for max_features is **7729** (the total number of features in our dataset, F = 7729).

#### Why tune max_features?
- **Diversity**: Lower max_features creates more diverse trees by limiting feature choices at each split
- **Decorrelation**: Reduces correlation between trees in the ensemble
- **Generalization**: Can prevent overfitting by introducing more randomness
- **Computation**: Lower values speed up training

#### n_estimators tradeoff:
- **Primary tradeoff**: Computation time vs. model performance
- **More trees**: Better performance (up to a point), but slower training and prediction
- **Overfitting**: Generally **NO** - more trees rarely cause overfitting in random forests
- **Why no overfitting**: Each tree is trained independently with randomness (bootstrap sampling + random feature selection), so adding more trees averages out individual tree errors without memorizing training data

### Overall Conclusions

From Table 3, we observe:

## How to Run This Notebook

### Prerequisites
Ensure the following packages are installed:
- numpy
- pandas
- scikit-learn
- matplotlib
- seaborn

### Execution Steps
1. **Run all cells sequentially** from top to bottom
2. **Grid searches** (cells with GridSearchCV) may take 2-10 minutes each
3. All results will be displayed inline and saved as images

### Reproducibility
- All random seeds are fixed (`random_state=101` for models, `np.random.seed(42)` for numpy)
- Running the notebook multiple times should produce identical results
- Data loading order is deterministic

### Output Files
- `model_comparison.png` - Comparison of all 4 models
- `feature_importance.png` - Top important features from best random forest
